# Model-2 (MLP - conventional CPET Variables)

Use the Python 3.11 kernel and private preprocessed data; see [setup](../README.md). Run cells top to bottom.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from ai_cpet_ots.dataloaders import load_CPET_data
from ai_cpet_ots.models import Model_2
from ai_cpet_ots.paths import CPET_CONTROL_DIR, CPET_OTS_DIR, MODEL_WEIGHTS, output_path
from ai_cpet_ots.testers import test_model_2

In [ ]:
def optimize_model(X_train, Y_train):
    """Search the notebook's study hyperparameter bounds using training inputs only.

    Returns the optimizer result used by train_model. Requires the optimization
    extra and can be expensive; it is not run in the default evaluation workflow."""
    from ai_cpet_ots.optimizers import Optimizer_Model_2

    opt = Optimizer_Model_2(X_train, Y_train)

    bounds = [
        {"name": "n_epochs", "type": "discrete", "domain": np.arange(5, 25, 5)},
        {"name": "dropout", "type": "discrete", "domain": np.arange(0, 1, 0.1)},
        {"name": "lr", "type": "discrete", "domain": (1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1)},
        {
            "name": "weight_decay",
            "type": "discrete",
            "domain": (0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1),
        },
        {"name": "batch_size", "type": "discrete", "domain": [16, 32, 64, 128]},
        {"name": "n_hidden_layers", "type": "discrete", "domain": np.arange(1, 6, 1)},
        {"name": "n_nodes", "type": "discrete", "domain": [16, 32, 64, 128]},
    ]

    _, opt_params_gpbo = opt.run_bo_gp(bounds, 10)

    return opt_params_gpbo

In [ ]:
def train_model(opt_params, X_train, Y_train):
    """Train this notebook's model from opt_params and the supplied training arrays.

    Keeps the original objective, optimizer, and epoch settings. Returns the
    trained model; checkpoint saving is controlled separately by save_model."""
    n_epochs, dropout, lr, weight_decay, batch_size, n_hidden_layers, n_nodes, threshold = (
        opt_params
    )

    n_epochs = int(n_epochs)
    batch_size = int(batch_size)
    n_hidden_layers = int(n_hidden_layers)
    n_nodes = int(n_nodes)

    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    # Convert to tensor
    x_train = torch.from_numpy(X_train).to(device)
    y_train = torch.from_numpy(Y_train).to(device)

    # Shuffle trainset and create minibatch
    trainset = TensorDataset(x_train, y_train)
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)

    # Initializing model
    model = Model_2(dropout, n_hidden_layers, n_nodes, threshold).to(device)

    pos_samples = np.sum(Y_train)
    neg_samples = len(Y_train) - pos_samples
    pos_weight = torch.tensor([neg_samples / pos_samples]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Training
    for e in range(n_epochs):
        train_loss = 0
        for batch in trainloader:
            x_batch, y_batch = batch
            x_batch, y_batch = x_batch.float().to(device), y_batch.to(device)

            y_pred, _ = model(x_batch)  # raw logits

            loss = criterion(y_pred.flatten(), y_batch)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        train_loss = loss.item() / len(trainloader)

        print(f"Epoch [{e + 1}/{n_epochs}] - Loss {train_loss:.4f}")

    return model

In [ ]:
def test_model(X_test, Y_test, model=None):
    """Load the published checkpoint when model is None and evaluate test inputs.

    Constructs the study DataLoader and forwards plot options to the reusable
    evaluator. Private inputs must use the original feature order and scaling."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    if model is None:
        model = Model_2().to(device)

        if device.type == "cpu":
            model.load_state_dict(
                torch.load(
                    MODEL_WEIGHTS / "Model_2.pth",
                    map_location=torch.device("cpu"),
                    weights_only=True,
                )
            )
        else:
            model.load_state_dict(torch.load(MODEL_WEIGHTS / "Model_2.pth", weights_only=True))

    testset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test))
    testloader = DataLoader(testset, batch_size=len(testset))
    auroc, auprc, f1, acc = test_model_2(model, "Model-2", testloader, True, True, True, True)
    print(
        f"AUROC {auroc:.3f}, AUCPR {auprc:.3f}, F1 {f1:.3f}, ACC {acc:.3f}, threshold {model.threshold}"
    )

In [ ]:
training_model = False
save_model = False
model_name = "Model_2"

In [ ]:
X, Y, X_train, Y_train, X_test, Y_test = load_CPET_data(CPET_CONTROL_DIR, CPET_OTS_DIR)

In [ ]:
if training_model:
    # Optimize model
    opt_params = optimize_model(X_train, Y_train)

    # Train the model
    model = train_model(opt_params, X_train, Y_train)

    # Test model
    test_model(X_test, Y_test, model)

    if save_model:
        torch.save(model.state_dict(), output_path(f"{model_name}.pth"))
else:
    test_model(X_test, Y_test)